# PART A: LIBARARY MANAGEMENT SYSTEM

### FUNCTIONS FOR LIBRARY MANAGEMENT

In [1]:
from datetime import datetime
from collections import Counter

"""
City Library Management System
================================
All data and logic live in this single module (no sub-modules).
Data is held in three in-memory stores:
  - BOOKS      : dict keyed by book_id
  - MEMBERS    : dict keyed by member_id
  - BORROW_LOG : list of transaction records
"""

from datetime import datetime
from collections import Counter

# ---------------------------------------------------------------------------
# In-memory data stores
# ---------------------------------------------------------------------------

BOOKS: dict = {}        # {book_id: {title, author, genre, availability}}
MEMBERS: dict = {}      # {member_id: {name, age, contact, borrowed_books}}
BORROW_LOG: list = []   # [{book_id, member_id, action, timestamp}]


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def _now() -> str:
    """Return current timestamp as a formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def reset_library() -> None:
    """Clear all in-memory data stores"""
    BOOKS.clear()
    MEMBERS.clear()
    BORROW_LOG.clear()

# ---------------------------------------------------------------------------
# Book Record Creation / Addition
# ---------------------------------------------------------------------------
def add_book(book_id: str, title: str, author: str, genre: str) -> dict:
    """
    Add a new book to the library.

    Parameters
    ----------
    book_id : unique identifier (e.g. 'B001')
    title   : book title
    author  : author name
    genre   : genre label (e.g. 'Fiction', 'Science')

    Returns
    -------
    The newly created book record, or raises ValueError if ID already exists.
    """
    if book_id in BOOKS:
        raise ValueError(f"Book ID '{book_id}' already exists.")

    book = {
        "book_id":      book_id,
        "title":        title.strip(),
        "author":       author.strip(),
        "genre":        genre.strip(),
        "availability": "Available",
    }
    BOOKS[book_id] = book
    return book

# ---------------------------------------------------------------------------
# Member Record Creation / Addition
# ---------------------------------------------------------------------------

def add_member(member_id: str, name: str, age: int, contact: str) -> dict:
    """
    Register a new library member.

    Parameters
    ----------
    member_id : unique identifier (e.g. 'M001')
    name      : full name
    age       : age in years (must be ≥ 5)
    contact   : email or phone number

    Returns
    -------
    The newly created member record, or 
        - raises ValueError if ID already exists
        - raises ValueError if age is less than 5
        - FUTURE: we can use a combination of name and contact to check for duplicates,
    """
    if member_id in MEMBERS:
        raise ValueError(f"Member ID '{member_id}' already exists.")
    if age < 5:
        raise ValueError("Member age must be at least 5.")

    member = {
        "member_id":      member_id,
        "name":           name.strip(),
        "age":            age,
        "contact":        contact.strip(),
        "borrowed_books": [],   # list of book_ids currently borrowed
    }
    MEMBERS[member_id] = member
    return member


# ---------------------------------------------------------------------------
# Borrow and Return System
# ---------------------------------------------------------------------------

def issue_book(member_id: str, book_id: str) -> dict:
    """
    Issue a book to a member.

    Business rules
    --------------
    - Book and member must exist.
    - Book must be 'Available'.
    - A member cannot borrow the same book twice simultaneously.

    Returns
    -------
    The borrow-log entry that was recorded.
    """
    member = get_member(member_id)
    book   = get_book(book_id)

    if book["availability"] != "Available":
        raise ValueError(
            f"Book '{book['title']}' is already issued and not available."
        )
    if book_id in member["borrowed_books"]:
        raise ValueError(
            f"Member '{member['name']}' already has book '{book['title']}'."
        )

    # Update state
    BOOKS[book_id]["availability"] = "Issued"
    MEMBERS[member_id]["borrowed_books"].append(book_id)

    entry = {
        "action":    "ISSUE",
        "member_id": member_id,
        "book_id":   book_id,
        "timestamp": _now(),
    }
    BORROW_LOG.append(entry)
    return entry


def return_book(member_id: str, book_id: str) -> dict:
    """
    Return a book previously borrowed by a member.

    Returns
    -------
    The borrow-log entry that was recorded.
    """
    member = get_member(member_id)
    book   = get_book(book_id)

    if book_id not in member["borrowed_books"]:
        raise ValueError(
            f"Member '{member['name']}' does not have book '{book['title']}'."
        )

    # Update state
    BOOKS[book_id]["availability"] = "Available"
    MEMBERS[member_id]["borrowed_books"].remove(book_id)

    entry = {
        "action":    "RETURN",
        "member_id": member_id,
        "book_id":   book_id,
        "timestamp": _now(),
    }
    BORROW_LOG.append(entry)
    return entry

def get_member(member_id: str) -> dict:
    """Return member record or raise KeyError."""
    if member_id not in MEMBERS:
        raise KeyError(f"Member ID '{member_id}' not found.")
    return MEMBERS[member_id]

def update_book_availability(book_id: str, status: str) -> dict:
    """
    Manually set the availability of a book.

    Parameters
    ----------
    book_id : target book
    status  : 'Available' or 'Issued'
    """
    if book_id not in BOOKS:
        raise KeyError(f"Book ID '{book_id}' not found.")
    if status not in ("Available", "Issued"):
        raise ValueError("Status must be 'Available' or 'Issued'.")
    BOOKS[book_id]["availability"] = status
    return BOOKS[book_id]

def get_book(book_id: str) -> dict:
    """Return book record or raise KeyError."""
    if book_id not in BOOKS:
        raise KeyError(f"Book ID '{book_id}' not found.")
    return BOOKS[book_id]

# ---------------------------------------------------------------------------
# Display Helpers 
# ---------------------------------------------------------------------------
def display_books(books: list):
    """Display all the books, related details and availability status."""
    if not books:
        print("  No records found.")
    for b in books:
        avail_tag = "[AVAILABLE]" if b["availability"] == "Available" else "[ISSUED]   "
        print(
            f"  {avail_tag} {b['book_id']} | {b['title']}"
            f" by {b['author']} | Genre: {b['genre']}"
        )

def display_members(members: list):
    """Display all the members and their details."""
    if not members:
        print("  No records found.")
    for m in members:
        borrowed = ", ".join(m["borrowed_books"]) if m["borrowed_books"] else "None"
        print(
            f"  {m['member_id']} | {m['name']} | Age: {m['age']} | Contact: {m['contact']}"
            f" | Borrowed Books: {borrowed}"
        )

In [23]:
reset_library()

# CREATE BOOK AND MEMBER RECORDS

### ADD BOOKS & TEST LIST

In [24]:
# Add individual books
import pandas as pd
books_data = pd.DataFrame([
    ("B001", "The Great Gatsby",        "F. Scott Fitzgerald", "Fiction"),
    ("B002", "1984",                    "George Orwell",       "Dystopian"),
    ("B003", "A Brief History of Time", "Stephen Hawking",     "Science"),
    ("B004", "Dune",                    "Frank Herbert",       "Science Fiction"),
    ("B005", "The Hobbit",              "J.R.R. Tolkien",      "Fantasy"),
], columns=["id", "title", "author", "genre"])

for _, row in books_data.iterrows():
    add_book(row["id"], row["title"], row["author"], row["genre"])

In [25]:
## LIST OF ALL AVAILABLE BOOKS
display_books(books = list(BOOKS.values()))

  [AVAILABLE] B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [AVAILABLE] B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy


In [26]:
## TESTING DUPLICATE BOOK ID

## Should fail
try:
    add_book("B001", "The Hitchhiker's Guide to the Galaxy", "Douglas Adams", "Science Fiction")
    print("Successfully added duplicate book ID 'B001' (this should fail).")
except ValueError as e:
    print(f"Error: {e}")

## Should succeed
try:
    add_book("B006", "The Hitchhiker's Guide to the Galaxy", "Douglas Adams", "Science Fiction")
    print("Successfully added book ID 'B006'.")
except ValueError as e:
    print(f"Error: {e}")

## Should succeed, multiple copies of the same book with different IDs
try:
    add_book("B007", "The Hitchhiker's Guide to the Galaxy", "Douglas Adams", "Science Fiction")
    print("Successfully added book ID 'B007'.")
except ValueError as e:
    print(f"Error: {e}")


Error: Book ID 'B001' already exists.
Successfully added book ID 'B006'.
Successfully added book ID 'B007'.


In [27]:
display_books(list(BOOKS.values()))

  [AVAILABLE] B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [AVAILABLE] B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
  [AVAILABLE] B006 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction
  [AVAILABLE] B007 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction


### ADD MEMBERS AND TEST LIST

In [28]:
members_data = pd.DataFrame([
    ("M001", "Alice Smith",   30, "alice@email.com"),
    ("M002", "Bob Johnson",   25, "bob@email.com"),
    ("M003", "Charlie Lee",   22, "charles@email.com"),
    ("M004", "Diana Prince", 28, "diana@email.com"),
    ("M005", "Ethan Hunt",   35, "ethan@email.com")
], columns=["id", "name", "age", "email"])

for _, row in members_data.iterrows():
    add_member(row["id"], row["name"], row["age"], row["email"])

In [29]:
display_members(list(MEMBERS.values()))

  M001 | Alice Smith | Age: 30 | Contact: alice@email.com | Borrowed Books: None
  M002 | Bob Johnson | Age: 25 | Contact: bob@email.com | Borrowed Books: None
  M003 | Charlie Lee | Age: 22 | Contact: charles@email.com | Borrowed Books: None
  M004 | Diana Prince | Age: 28 | Contact: diana@email.com | Borrowed Books: None
  M005 | Ethan Hunt | Age: 35 | Contact: ethan@email.com | Borrowed Books: None


In [30]:
## Catch age related error for member creation
try:
    add_member("M099", "Child", 3, "child@example.com")
except ValueError as e:
    print(f"Expected error caught: {e}")

## Catch member ID duplication error
try:
    add_member("M001", "Child", 3, "child@example.com")
except ValueError as e:
    print(f"Expected error caught: {e}")

## Catch member ID duplication error
try:
    add_member("M006", "Fiona Glenanne", 27, "fiona@email.com")
    print("Successfully added member ID 'M006'.")
except ValueError as e:
    print(f"Expected error caught: {e}")

display_members(list(MEMBERS.values()))

Expected error caught: Member age must be at least 5.
Expected error caught: Member ID 'M001' already exists.
Successfully added member ID 'M006'.
  M001 | Alice Smith | Age: 30 | Contact: alice@email.com | Borrowed Books: None
  M002 | Bob Johnson | Age: 25 | Contact: bob@email.com | Borrowed Books: None
  M003 | Charlie Lee | Age: 22 | Contact: charles@email.com | Borrowed Books: None
  M004 | Diana Prince | Age: 28 | Contact: diana@email.com | Borrowed Books: None
  M005 | Ethan Hunt | Age: 35 | Contact: ethan@email.com | Borrowed Books: None
  M006 | Fiona Glenanne | Age: 27 | Contact: fiona@email.com | Borrowed Books: None


# BORROW & RETURN SYSTEM

In [31]:
## MEMBER BORROWS MORE THAN ONE BOOK
book_id = "B001"
member_id = "M001"
try:
    issue_book(book_id=book_id, member_id=member_id)  # Alice borrows "A Brief History of Time"
    print(f"Member '{MEMBERS[member_id]['name']}' successfully borrowed book '{BOOKS[book_id]['title']}'.")
except ValueError as e:
    print(f"Error : {e}  to {get_member(member_id)['name']}")

book_id = "B004"
member_id = "M001"
try:
    issue_book(book_id=book_id, member_id=member_id)  # Alice borrows "Dune"
    print(f"Member '{MEMBERS[member_id]['name']}' successfully borrowed book '{BOOKS[book_id]['title']}'.")
except ValueError as e:
    print(f"Error: {e} to {get_member(member_id)['name']}")

book_id = "B003"
member_id = "M002"
try:
    issue_book(book_id=book_id, member_id=member_id)  # Bob tries to borrow "A Brief History of Time" which is already issued to Alice
    print(f"Member '{MEMBERS[member_id]['name']}' successfully borrowed book '{BOOKS[book_id]['title']}'.")
except ValueError as e:
    print(f"Error: {e} to {get_member(member_id)['name']}")

Member 'Alice Smith' successfully borrowed book 'The Great Gatsby'.
Member 'Alice Smith' successfully borrowed book 'Dune'.
Member 'Bob Johnson' successfully borrowed book 'A Brief History of Time'.


In [32]:
display_books(list(BOOKS.values()))

  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [ISSUED]    B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [ISSUED]    B004 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
  [AVAILABLE] B006 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction
  [AVAILABLE] B007 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction


In [33]:
## RETURN A BOOK
book_id = "B001"
member_id = "M001"

try:
    return_book(book_id=book_id, member_id=member_id)  # Alice returns "A Brief History of Time"
except ValueError as e:
    print(f"Error: {e}")

In [34]:
display_books(list(BOOKS.values()))

  [AVAILABLE] B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [ISSUED]    B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [ISSUED]    B004 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
  [AVAILABLE] B006 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction
  [AVAILABLE] B007 | The Hitchhiker's Guide to the Galaxy by Douglas Adams | Genre: Science Fiction


# REPORT / QUERIES

In [35]:
## LIST ALL AVAILABLE BOOKS IN A GENRE
genre = "Science"
print(f"\nBooks in genre '{genre}':")
display_books(books = [b for b in BOOKS.values() if b["genre"] == genre])


Books in genre 'Science':
  [ISSUED]    B003 | A Brief History of Time by Stephen Hawking | Genre: Science


In [36]:
# SESARCH BOOKS BY TITLE
title = "Dune"
print(f"\nBooks with title '{title}':")
display_books(books = [b for b in BOOKS.values() if b["title"] == title])


Books with title 'Dune':
  [ISSUED]    B004 | Dune by Frank Herbert | Genre: Science Fiction


In [37]:
# SEARCH BOOKS BY AUTHOR
author = "George Orwell"
print(f"\nBooks by author '{author}':")
display_books(books = [b for b in BOOKS.values() if b["author"] == author])


Books by author 'George Orwell':
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian


In [38]:
## SHOW ALL BORROWED BOOKS
displa_borrow_log = pd.DataFrame(BORROW_LOG)
print("\nBorrow Log:")
if displa_borrow_log.empty:
    print("  No transactions recorded.")
else:
    print(displa_borrow_log.to_string(index=False))


Borrow Log:
action member_id book_id           timestamp
 ISSUE      M001    B001 2026-02-28 11:56:55
 ISSUE      M001    B004 2026-02-28 11:56:55
 ISSUE      M002    B003 2026-02-28 11:56:55
RETURN      M001    B001 2026-02-28 11:57:25


In [39]:
# LIST MEMBERS WHO HAVE BORROWED BOOKS
borrowers = set(entry["member_id"] for entry in BORROW_LOG if entry["action"] == "ISSUE")
print("\nMembers who have borrowed books:")
if not borrowers:
    print("  No members have borrowed books.")
else:
    for member_id in borrowers:
        member = get_member(member_id)
        print(f"  {member['member_id']} | {member['name']} | Contact: {member['contact']}")


Members who have borrowed books:
  M001 | Alice Smith | Contact: alice@email.com
  M002 | Bob Johnson | Contact: bob@email.com


In [40]:
# Display the most popular genre
if not BORROW_LOG:
    print("\nNo borrow transactions to analyze for popular genre.")
else:
    issued_books = [entry["book_id"] for entry in BORROW_LOG if entry["action"] == "ISSUE"]
    if not issued_books:
        print("\nNo books have been issued yet to determine popular genre.")
    else:
        genre_counts = Counter(BOOKS[book_id]["genre"] for book_id in issued_books)
        most_common_genre, count = genre_counts.most_common(1)[0]
        print(f"\nMost popular genre: '{most_common_genre}' with {count} issues.")


Most popular genre: 'Fiction' with 1 issues.
